In [1]:
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Point, LineString
import geopandas as gpd
import osmnx as ox
from typing import List

# Import necessary libraries
import pandas as pd
from collections import defaultdict
import geopandas as gpd
from shapely import wkt
import shapely

import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import pickle
%matplotlib inline
import datetime
import time
import numpy as np
import xml.etree.ElementTree as ET 
import matsim

import utm
from shapely.geometry import Polygon, Point
import gzip
from matplotlib import cm

import matplotlib.ticker as ticker
import matplotlib.font_manager as font_manager
import matplotlib as mpl
from matplotlib.lines import Line2D
from tqdm import tqdm
from matplotlib.colors import LinearSegmentedColormap
import json



In [2]:
with open('input/regionclusters.pkl', 'rb') as pickle_file: # liegt in ikg-useful-share/USEfUL XT/Modul 3/Auswertung-Oskar/input/
    regionclusters = pd.read_pickle(pickle_file)

def determine_area_type(raumtyp):
    if raumtyp in [1, 2, 3]:
        return 'Urban'
    elif raumtyp in [4, 5, 6]:
        return 'Suburban'
    elif raumtyp in [7, 8]:
        return 'Rural'
    else:
        return 'unknown'
    
category_dict = {
    "1": "Metropolitan Center",
    "2": "High-Density Residential Use",
    "3": "Dense Mixed Use",
    "4": "Residential Use",
    "5": "Industrial Use",
    "6": "Urbanized Periphery",
    "7": "Rural with Industrial Influence",
    "8": "Rural without Industrial Influence"
}


# Wende die Funktion auf die Spalte raumtyp an und erstelle die neue Spalte area_type
regionclusters['area_type_agg'] = regionclusters['raumtyp'].apply(determine_area_type)
regionclusters['area_type'] = regionclusters['raumtyp'].astype(str).map(category_dict)

# Explode the multipolygons into individual polygons
regionclusters_split = regionclusters.explode(index_parts=False)

# Reset index to clean up the DataFrame
regionclusters_split.reset_index(drop=True, inplace=True)

# Display the resulting DataFrame
regionclusters_split

# Read the CSV file
folder = "input/"
areas = pd.read_csv( folder + "plz_areas.csv")  

# Convert the 'WKT' column to Shapely MultiPolygon geometry
areas['geometry'] = areas['WKT'].apply(lambda wkt_str: wkt.loads(wkt_str))

# Create a GeoDataFrame
gdf_areas = gpd.GeoDataFrame(areas, geometry='geometry')

# Set the coordinate reference system (CRS)
gdf_areas.crs = 'EPSG:25832'  # Set the appropriate CRS if it's different


# Import Network
# MATSim network als Geopandas dataframe inkl. LINESTRINGS einlesen
# network = matsim.read_network('D:\\Hannover Daten\\MatSim\\Network XT\\car_network_encfix.xml.gz').as_geo().set_crs('epsg:25832') # liegt in ikg-useful-share/USEfUL XT/Modul 3/Auswertung-Oskar/input/
# network = matsim.read_network('input/simRes/500-it/basecase_13052025/basecase_13052025.output_network.xml.gz').as_geo().set_crs('epsg:25832')
network = matsim.read_network('input/simRes/BC_500_it_no_reduction/basecase_13052025.output_network.xml.gz').as_geo().set_crs('epsg:25832')
# dict mit Netzwerk Link Längen
# link_length = network[['link_id', 'length']].set_index('link_id').to_dict()['length']
network.head()

# falls vorhanden Ergebnisse der nächsten Zellen laden, sonst überspringen
with open('input/regionclusters.pkl', 'rb') as pickle_file: # liegt in ikg-useful-share/USEfUL XT/Modul 3/Auswertung-Oskar/input/
    regionclusters = pd.read_pickle(pickle_file)
with open('input/networkplus.pkl', 'rb') as pickle_file: # liegt in ikg-useful-share/USEfUL XT/Modul 3/Auswertung-Oskar/input/
    networkplus = pd.read_pickle(pickle_file)     




C:\Users\bienzeisler\AppData\Local\Programs\Python\Python313\Lib\pickle.py:1760: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  setstate(state)
C:\Users\bienzeisler\AppData\Local\Programs\Python\Python313\Lib\pickle.py:1760: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  setstate(state)


In [3]:
# Create a color column based on freespeed and lane conditions
def assign_type(row):
    if row['freespeed_kmh'] <= 50:
        return 'urban'
    elif row['freespeed_kmh'] > 50 and row['permlanes'] >= 2:
        return 'highway'
    else:
        return 'rural'  # or any other default color

network['freespeed_kmh'] = network['freespeed'] * 3.6
network['road_type'] = network.apply(assign_type, axis=1)

link_length = network[['link_id', 'length']].set_index('link_id').to_dict()['length']
link_geometry = network[['link_id', 'geometry']].set_index('link_id').to_dict()['geometry']
link_type = network[['link_id', 'road_type']].set_index('link_id').to_dict()['road_type']
# dict mit Netzwerk Link Raumtypen
link_raumtyp = networkplus['raumtyp'].to_dict()

# Create a dictionary linking each area type code to its description
area_type_dict = {
    1: "Metropolitan Center",
    2: "High-Density Residential Use",
    3: "Dense Mixed Use",
    4: "Residential Use",
    5: "Industrial Use",
    6: "Urbanized Periphery",
    7: "Rural with Industrial Influence",
    8: "Rural without Industrial Influence"
}

from matplotlib.colors import to_hex
# Define the desired order of legend labels in English
# Create a dictionary linking each category to a color



category_color_dict = {
    "Metropolitan Center": "#482878",
    "High-Density Residential Use": "#3f4989",
    "Dense Mixed Use": "#31688e",
    "Residential Use": "#26828e",
    "Industrial Use": "#1f9e89",
    "Urbanized Periphery": "#35b779",
    "Rural with Industrial Influence": "#6fce58",
    "Rural without Industrial Influence": "#b5de2b"
}



category_color_dict_num = {
    0: "#482878",
    1: "#3f4989",
    2: "#31688e",
    3: "#26828e",
    4: "#1f9e89",
    5: "#35b779",
    6: "#6fce58",
    7: "#b5de2b"
}



# Create a dictionary linking each category to a color
category_color = {category: i / (len(category_color_dict) - 1) for i, category in enumerate(category_color_dict)}

# Create the colormap
cmap = LinearSegmentedColormap.from_list("custom_cmap", sns.color_palette("viridis_r", len(category_color_dict))[::-1])

# Convert the colormap to a dictionary
palette_dict = {cat: to_hex(cmap(category_color[cat])) for cat in category_color_dict}


In [4]:
# def Methoden

def parse_events(event_file):
    """
    This function parses events from a given event file. It filters out events of type 'left link' and 'actstart'.
    It also counts the number of events for different types of vehicles and stores the last link and time for each vehicle.
    """
    
    # Filter events of interest
    events = matsim.event_reader(event_file, types='left link,actstart')

    # Consolidated link counts and vehicle tours
    link_counts = defaultdict(lambda: defaultdict(int))
    vehicle_tour = defaultdict(list)  # This will now store (link, time) tuples
    service_events = defaultdict(list)
    last_link = defaultdict(str)
    
    # Helper function to update link counts
    def update_link_counts(vehicle_type, link, time):
        if link != last_link[vehicle]:
            link_counts[link][vehicle_type] += 1
            vehicle_tour[vehicle].append((link, time))  # Store the link and time as a tuple
            last_link[vehicle] = link
    
    nr_events = 0
    for event in events:        
        if event['time'] > DAYEND: continue
            
        nr_events =  nr_events + 1
        
        if event['type'] == 'left link':
            vehicle = event['vehicle']
            if '_Supply_Vehicle_' in vehicle or '_veh_supply_' in vehicle:
                if "supply_light_van" in vehicle:
                    update_link_counts('supply_van', event['link'], event['time'])
                elif "light" in vehicle:
                    update_link_counts('truck_light_count', event['link'], event['time'])
                else:
                    update_link_counts('truck_count', event['link'], event['time'])

            elif '_CEP_Vehicle_' in vehicle or '_veh_cep_' in vehicle or '_egrocery_van_' in vehicle:
                update_link_counts('van_count', event['link'], event['time'])
                if "size_m" in vehicle:
                    update_link_counts('m_count', event['link'], event['time'])
                elif "size_xl" in vehicle:
                    update_link_counts('xl_count', event['link'], event['time'])
            elif '_cargoBike_' in vehicle or '_cargobike_' in vehicle:
                update_link_counts('bike_count', event['link'], event['time'])
            
        elif event['type'] == 'actstart' and event['actType'] in ['end', 'service']:
            person = event['person']
            if event['actType'] == 'end':
                if any(keyword in person for keyword in ['_Supply_Vehicle_', '_veh_supply_', '_CEP_Vehicle_', '_veh_cep_', '_egrocery_van_', '_cargoBike_', '_cargobike_']):
                    vehicle_tour[person].append((event['link'], event['time']))  # Store the link and time as a tuple
            if event['actType'] == 'service':
                service_events[person].append(event['link'])  # No need to store time for service events as per your original code
                
    
    # Convert link counts to DataFrame and handle missing values
    df = pd.DataFrame(link_counts).transpose()
    df.fillna(0, inplace=True)

    # Ensure columns exist in DataFrame
    for col in ['truck_count', 'truck_light_count', "supply_van_count", 'van_count', 'bike_count']:
        if col not in df.columns:
            df[col] = 0
     
    # Explicitly ensure the 'bike_count' column exists
    if 'bike_count' not in df.columns:
        df['bike_count'] = 0
        
    df['total_count'] = df['truck_count'] + df['truck_light_count'] + df['supply_van_count'] + df['van_count'] + df['bike_count']
        
    df['total_count_supply'] = df['truck_count'] + df['truck_light_count'] + df['supply_van_count'] 
    
    network_volumes = network.merge(df, left_on='link_id', right_index=True, how='left')
    network_volumes.fillna(0, inplace=True)
    
    return vehicle_tour, service_events, network_volumes, nr_events

# vehicle string pattern
# def parse_events(event_file):
#     """
#     This function parses events from a given event file. It filters out events of type 'left link' and 'actstart'.
#     It also counts the number of events for different types of vehicles and stores the last link for each vehicle.
    
#     Args:
#         event_file (str): The path to the event file to be parsed.
        
#     Returns:
#         vehicle_tour (dict): A dictionary mapping each vehicle to a list of links in its tour.
#         service_events (dict): A dictionary mapping each person to a list of service events.
#         network_volumes (DataFrame): A DataFrame containing the counts of each type of vehicle on each link.
#     """

#     # Helper function to update link counts
#     def update_link_counts(vehicle_type, link, time):
#         if link != last_link[vehicle]:
#             link_counts[link][vehicle_type] += 1
#             vehicle_tour[vehicle].append((link, time))  # Store the link and time as a tuple
#             last_link[vehicle] = link
        
#     # Only returns events of type 'left link' and 'actstart:
#     events = matsim.event_reader(event_file, types='left link,actstart')

#     # defaultdict creates a blank dict entry on first reference; similar to {} but more friendly
#     link_counts_trucks = defaultdict(int)
#     link_counts_vans = defaultdict(int)
#     link_counts_bikes = defaultdict(int)
#     link_counts_cars = defaultdict(int)
#     link_counts = defaultdict(lambda: defaultdict(int))

#     vehicle_tour = defaultdict(list)
#     service_events = defaultdict(list)
#     last_link = defaultdict(str)

#     nr_events = 0
#     for event in events:        
#         if event['time'] > DAYEND: continue
            
#         nr_events =  nr_events + 1
        
#         if event['type'] == 'left link':
#             vehicle = event['vehicle']
#             if '_Supply_Vehicle_' in vehicle or '_veh_supply_' in vehicle:
#                 if "supply_light_van" in vehicle:
#                     update_link_counts('supply_van', event['link'], event['time'])
#                 elif "light" in vehicle:
#                     update_link_counts('truck_light_count', event['link'], event['time'])
#                 else:
#                     update_link_counts('truck_count', event['link'], event['time'])

#             elif '_CEP_Vehicle_' in vehicle or '_veh_cep_' in vehicle or '_egrocery_van_' in vehicle:
#                 update_link_counts('van_count', event['link'], event['time'])
#                 if "size_m" in vehicle:
#                     update_link_counts('m_count', event['link'], event['time'])
#                 elif "size_l" in vehicle:
#                     update_link_counts('xl_count', event['link'], event['time'])
#             elif '_cargoBike_' in vehicle or '_cargobike_' in vehicle:
#                 update_link_counts('bike_count', event['link'], event['time'])
            
#         elif event['type'] == 'actstart' and event['actType'] in ['end', 'service']:
#             person = event['person']
#             if event['actType'] == 'end':
#                 if any(keyword in person for keyword in ['_Supply_Vehicle_', '_veh_supply_', '_CEP_Vehicle_', '_veh_cep_', '_egrocery_van_', '_cargoBike_', '_cargobike_']):
#                     vehicle_tour[person].append((event['link'], event['time']))  # Store the link and time as a tuple
#             if event['actType'] == 'service':
#                 service_events[person].append(event['link'])  # No need to store time for service events as per your original code
    


#     # convert link_counts dict to a pandas dataframe
#     link_counts_trucks = pd.DataFrame.from_dict(link_counts_trucks, orient='index', columns=['truck_count']).rename_axis('link_id')
#     link_counts_vans = pd.DataFrame.from_dict(link_counts_vans, orient='index', columns=['van_count']).rename_axis('link_id')
#     link_counts_bikes = pd.DataFrame.from_dict(link_counts_bikes, orient='index', columns=['bike_count']).rename_axis('link_id')
#     link_counts_cars = pd.DataFrame.from_dict(link_counts_cars, orient='index', columns=['car_count']).rename_axis('link_id')

#     # attach counts to our Geopandas network from above
#     network_volumes = network.merge(link_counts_trucks, on='link_id', how='left').merge(link_counts_vans, on='link_id', how='left').merge(link_counts_bikes, on='link_id', how='left').merge(link_counts_cars, on='link_id', how='left')
#     network_volumes['car_count'] = network_volumes['car_count'].fillna(0)
#     network_volumes['truck_count'] = network_volumes['truck_count'].fillna(0)
#     network_volumes['van_count'] = network_volumes['van_count'].fillna(0)
#     network_volumes['bike_count'] = network_volumes['bike_count'].fillna(0)
#     network_volumes['total_count'] = network_volumes['truck_count'] + network_volumes['van_count'] + network_volumes['bike_count']

#     return vehicle_tour, service_events, network_volumes, nr_events


def vehicle_stats(vehicle_tour, service_events):
    """
    This function calculates statistics for each vehicle, including the number of services, total tour length,
    distance to the first service, and distribution of services over different types of areas.
    
    Args:
        vehicle_tour (dict): A dictionary mapping each vehicle to a list of links in its tour.
        service_events (dict): A dictionary mapping each person to a list of service events.
        
    Returns:
        veh_df (DataFrame): A DataFrame containing statistics for each vehicle.
    """
        
    # init Ergebnis Objekte
    vehicle_stats = list()

    # Loop über Fahrzeuge und ihre Services
    for veh, ser in service_events.items():
        # Unterscheidung Fahrzeugtypen
        if '_Supply_Vehicle_' in veh or '_veh_supply_' in veh:
            c = 'truck'            
        if '_CEP_Vehicle_' in veh or '_veh_cep_' in veh or '_egrocery_van_' in veh:
            c = 'van'
        if '_cargoBike_' in veh or '_cargobike_' in veh:
            c = 'bike'
        
        # tourlen = 0
        # firstservicedist = None
        # # Loop über alle Tourlinks des Fahrzeuges
        # for link in vehicle_tour[veh]:
        #     # falls erster Service erreicht, Strecke bis dahin speichern
        #     if link == ser[0] and firstservicedist is None:
        #         firstservicedist = tourlen
        #     # Streckenlänge aufsummieren
        #     tourlen += link_length[link]
        # # Sonderfälle abfangen, wo ein Service kurz vor Tagesende beginnt und am nächsten Tag weitergefahren wird
        # if firstservicedist is None: firstservicedist = tourlen

        tourlen = 0
        firstservicedist = None
        service_dists = dict()

        if not ser:
            continue

        for link, time in vehicle_tour[veh]: 
            # Strecke aufsummieren
            link_km = link_length[link] / 1000  # Direkt km
            tourlen += link_km
            for s in ser:
                # Sobald Servicepunkt erreicht (einmalig erfassen)
                if link == s and s not in service_dists:
                    service_dists[s] = tourlen
                    if firstservicedist is None:
                        firstservicedist = tourlen  # Speichere ersten Treffer

        # Fallback: falls ein Service nicht in der Tour auftaucht
        for s in ser:
            if s not in service_dists:
                service_dists[s] = tourlen
        if firstservicedist is None:
            firstservicedist = tourlen
           
        indexService = 0
        start_service_distance_count = False
        currentServiceDist = 0
        serviceDistList= []    

        # Service Verteilung über Raumtypen ermitteln
        raumtypen_services = defaultdict(int)
        for s in ser:
            rt = link_raumtyp.get(s, 0) # keinem Raumtyp zugeordnet -> 0
            raumtypen_services[rt] += 1        
    

        summ_service_distances = 0
        previous_time = 0     


        vehicle_stats.append([
            veh, c, len(ser),
            tourlen ,
            firstservicedist ,
            raumtypen_services,
            service_dists,
            firstservicedist   # explizite Spalte für initial_delivery_distance
        ])


    # Dataframe aus Liste
    veh_df = pd.DataFrame(vehicle_stats, columns=[
        'vehicle_id', 'veh_class', 'service_num',
        'tour_km', 'first_service_dist',
        'raumsplit', 'service_dists',
        'initial_delivery_distance'
    ])

    return veh_df

# Definition of help-methods needed for the convertion process

# Method returning the index of an element of a dictionary
def get_nth_key(dictionary, n=0):
    """
    This function returns the nth key of a dictionary.
    
    Args:
        dictionary (dict): The dictionary to get the key from.
        n (int): The index of the key to get. Default is 0.
        
    Returns:
        key: The nth key of the dictionary.
    """
        
    if n < 0:
        n += len(dictionary)
    for i, key in enumerate(dictionary.keys()):
        if i == n:
            return key
    raise IndexError("dictionary index out of range") 

    
# Get Logistic Provider from XML ID
def getProviderFromID(carrierID):
    """
    This function returns the provider name based on the carrier ID.
    
    Args:
        carrierID (str): The ID of the carrier.
        
    Returns:
        str: The name of the provider.
    """
        
    if("dhl" in str(carrierID)):
        return "dhl"  
    elif("amazon" in str(carrierID)):
        return "amazon"  
    elif("ups" in str(carrierID)):
        return "ups"  
    elif("gls" in str(carrierID)):
        return "gls"  
    elif("dpd" in str(carrierID)):
        return "dpd"  
    elif("fedex" in str(carrierID)):
        return "fedex"  
    elif("hermes" in str(carrierID)):
        return "hermes"  
    elif("wl" in str(carrierID)):
        return "White-Label"  
    else:
        raise ValueError('Carrier Provider unknown: ' + str(carrierID))
        
# Definition of needen Classes: Carrier, Vehicle, Plan & Service with Variables

# A carrier object has an ID and dictionaries with all Vehicles and all Services
class Carrier:  
    """
    This class represents a Carrier with an ID and dictionaries with all Vehicles and all Services.
    """
        
    def __init__(self, carrierID): 
        self.carrierID = carrierID 
        self.vehicles = {} 
        self.services = {}   
        self.missedDeliveries = []
        self.logisticProvider = getProviderFromID(self.carrierID)
        
    def __str__(self):
        return "[Carrier ID: " + str(self.carrierID) + " with " + str(len(self.vehicles)) + " Vehicles and " + str(len(self.services)) + " Services]"
    
    def __repr__(self):
        return self.__str__()
    
    def getNumberOfVehicles(self):
        return (len(self.vehicles))
    
    def getNumberOfServices(self):
        return (len(self.services))
        
    def addVehicle(self, vehicle): 
        self.vehicles[vehicle.getVehicleId] = vehicle
        
    def addService(self, service): 
        self.services[service.getServiceID] = service

    def getTotalNumberofServices(self):  
        return sum((s.getDemand() for s in self.services.values()))
    
    def getProvider(self):
        return self.logisticProvider
    
    def getServices(self): 
        return [s for s in self.services.values()]   
    
    def getCarrierId(self): 
        return self.carrierID    
    
    def getVehicles(self): 
        return self.vehicles
    
    def getMissedDeliveries(self): 
        return self.missedDeliveries  
    
    def setMissedDeliveries(self, missDeliveries): 
        self.missedDeliveries = missDeliveries


# A Vehicle has an ID, a Typ and you can add Plans to this vehicles with services and Routes. 
# These Vehicles / Plans have to be converted to Sumo! 
class Vehicle: 
    """
    This class represents a Vehicle with an ID, a Type and you can add Plans to this vehicles with services and Routes.
    """
        
    def __init__(self, vehicleID, vehicleType): 
        self.vehicleID = vehicleID 
        self.vehicleType = vehicleType 
        self.plans = []
        
    def __str__(self):
        return "[Vehicle ID: " + str(self.vehicleID) + " with Type: " + self.vehicleType +"]"
    
    def __repr__(self):
        return self.__str__()
    
    def addVehiclePlan(self, plan): 
        self.plans.append(plan)
        
    def changeVehicleType(self, vehicleType): 
        self.vehicleType == vehicleType
        
    def getVehicleId(self): 
        return self.vehicleID
    
    def getPlans(self):
        return self.plans


# A Plan is a sequence of Activies (start, service, service, ... , end)
# all important Information are stored in the activities / legs dictionary
# The Order of the sequence is stored as a list called "planSequence"
class Plan:  
    """
    This class represents a Plan which is a sequence of Activities (start, service, service, ... , end).
    """
        
    def __init__(self, planId, vehicle, activities, legs): 
        self.planId = planId
        self.vehicle = vehicle
        self.activities = activities 
        self.legs = legs 
        self.planSequence = [] 
        
    def __str__(self):
        return "[Plan ID: " + str(self.planId) + " with " + str(len(self.activities)) +" Activities and " + str(len(self.legs)) + " Legs]"
    
    def __repr__(self):
        return self.__str__()
    
    def getPlanSequence(self):
        return self.planSequence
    
    def createInternalPlanSequence(self):
        for i in range (len(self.legs)):            
            self.planSequence.append(get_nth_key(self.activities, i))
            self.planSequence.append(get_nth_key(self.legs, i))
        self.planSequence.append(get_nth_key(self.activities, len(self.activities)-1))    

# A Service is a data Container storing all availble Infomation from the CSV-File
class Service(): 
    """
    This class represents a Service which is a data Container storing all available Information from the CSV-File.
    """
        
    def __init__(self, serviceType , serviceID, capacityDemand, duration, link, extra_attributes=None):
        self.serviceType = serviceType
        self.serviceID = serviceID 
        self.capacityDemand = capacityDemand 
        self.duration = duration 
        self.link = link 
        self.extra_attributes = extra_attributes or {}
        
    def __str__(self):
        return "[Service ID: " + str(self.serviceID) + "(Type: " + str(self.serviceType) + ") with a CapacityDemand of " + str(self.capacityDemand) +" to Link: " + str(self.link) + "]"
    
    def __repr__(self):
        return self.__str__()

    def getAttribute(self, key, default=None):
        return self.extra_attributes.get(key, default)
    
    def getServiceType(self): 
        return self.serviceType
        
    def changeVehicleType(self, vehicleType): 
        self.vehicleType == vehicleType
        
    def getDemand(self): 
        return self.capacityDemand
        
    def getServiceID(self): 
        return self.serviceID
    
    def getServiceLink(self): 
        return self.link
    
def extract_provider(vehicle_id):
    """
    This function extracts the provider from the vehicle ID.
    
    Args:
        vehicle_id (str): The ID of the vehicle.
        
    Returns:
        str: The name of the provider.
    """
        
    return vehicle_id.split("_")[1]

def extract_veh_size(vehicle_id):
    """
    Extracts the vehicle size from the vehicle ID by looking for 'size_' and returning the next token.
    
    Args:
        vehicle_id (str): The ID of the vehicle.
        
    Returns:
        str: The vehicle size (e.g., 'l', 'm', 'l'), or 'unknown' if not found.
    """
    try:
        parts = vehicle_id.split("size_")
        if len(parts) > 1:
            return parts[1].split("_")[0]
        else:
            return "unknown"
    except Exception as e:
        print(f"Error extracting vehicle size from '{vehicle_id}': {e}")
        return "unknown"

def extract_main_area_type(raumsplit):
    """
    This function extracts the main area type from the raumsplit.
    
    Args:
        raumsplit (dict): The dictionary containing the raumsplit.
        
    Returns:
        str: The main area type.
    """
        
    return max(raumsplit.items(), key=lambda x: x[1])[0]

def process_vehicle_data(veh_df_van):
    """
    This function processes the vehicle data and adds provider, vehicle size, and main area type to the DataFrame.
    
    Args:
        veh_df_van (DataFrame): The DataFrame containing the vehicle data.
        
    Returns:
        DataFrame: The processed DataFrame.
    """
        
    veh_df_van['provider'] = veh_df_van['vehicle_id'].apply(extract_provider)
    veh_df_van['veh_size'] = veh_df_van['vehicle_id'].apply(extract_veh_size)
    veh_df_van['main_area_type'] = veh_df_van['raumsplit'].apply(extract_main_area_type)

    return veh_df_van

def get_vehicles(vehicle_tour):
    """
    This function gets the vehicles from the vehicle tour.
    
    Args:
        vehicle_tour (dict): The dictionary containing the vehicle tour.
        
    Returns:
        list: The list of vehicles.
    """
        
    return [vehicle for vehicle in vehicle_tour if "_supply_" not in vehicle]

def create_plot_data(vehicles, event_file):
    """
    This function creates the plot data.
    
    Args:
        vehicles (list): The list of vehicles.
        event_file (str): The path to the event file.
        
    Returns:
        DataFrame: The DataFrame containing the plot data.
    """
        
    plot_data = pd.DataFrame({
    "Start time" : [0],
    "End time" : [0],
    "Tour Duration" : [0],
    "Service Duration" : [0],
    "Travel Duration" : [0],
    }, index=[vehicles,])

    #calculate start and end time
    events = matsim.event_reader(event_file, types='actstart,actend')
    event_lists = { 'start': [], 'end': [] }

    for event in events:
        if "_supply_" in event["person"]:
            continue
        if event['actType'] == "start" and event['type'] == "actend":
            event_lists['start'].append(event)
        if event['actType'] == "end" and event['type'] == "actstart":
            event_lists['end'].append(event)
    df_start = pd.DataFrame(event_lists['start'])
    df_end = pd.DataFrame(event_lists['end'])

    for event in range(len(df_start)):
            plot_data["Start time"][df_start.iloc[event]["person"]] = df_start.iloc[event]['time']
    for event in range(len(df_end)):
            plot_data["End time"][df_end.iloc[event]["person"]] = df_end.iloc[event]['time']


    #calculate travel time - need to remove the service time
    events = matsim.event_reader(event_file, types='actstart,actend')
    event_lists = { 'start': [], 'end': [] }

    for event in events:
        if "_supply_" in event["person"]:
            continue
        if event['actType'] == "service" and event['type'] == "actend":
            event_lists['start'].append(event)
        if event['actType'] == "service" and event['type'] == "actstart":
            event_lists['end'].append(event)
    df_start = pd.DataFrame(event_lists['start'])
    df_end = pd.DataFrame(event_lists['end'])

    for row in range(len(plot_data)):
        plot_data.iloc[row]["Tour Duration"] = plot_data.iloc[row]["End time"] - (plot_data.iloc[row]["Start time"])

    startG = df_start.groupby("person")
    endG = df_end.groupby("person")

    for n,g in startG:
        time = 0

        g = g.reset_index(drop=True)   

        g2 = endG.get_group(n)
        g2 = g2.reset_index(drop=True)

        result = pd.merge(g, g2, left_index=True, right_index=True)

        for i in range(len(result)):
            time = time + (result.loc[i]["time_y"] - result.loc[i]["time_x"])

            plot_data.loc[n, "Service Duration"] = abs(time)

    for row in range(len(plot_data)):
        plot_data.iloc[row]["Travel Duration"] = plot_data.iloc[row]["Tour Duration"] - (plot_data.iloc[row]["Service Duration"])

    plot_data = plot_data.astype(str)
    # for row in range(len(plot_data)):
    #     plot_data.iloc[row]["Start time"] = time.strftime("%H:%M:%S", time.gmtime(int(plot_data.iloc[row]["Start time"])))
    #     plot_data.iloc[row]["End time"] = time.strftime("%H:%M:%S", time.gmtime(int(plot_data.iloc[row]["End time"])))
    #     plot_data.iloc[row]["Tour Duration"] = time.strftime("%H:%M:%S", time.gmtime(int(plot_data.iloc[row]["Tour Duration"])))
    #     plot_data.iloc[row]["Service Duration"] = time.strftime("%H:%M:%S", time.gmtime(int(plot_data.iloc[row]["Service Duration"])))
    #     plot_data.iloc[row]["Travel Duration"] = time.strftime("%H:%M:%S", time.gmtime(int(plot_data.iloc[row]["Travel Duration"])))



    plot_data.reset_index(inplace=True)
    plot_data = plot_data.rename(columns={'level_0': 'vehicle_id'})

    plot_data["Start time formatted"] = pd.to_datetime(plot_data['Start time'], unit='s')
    plot_data["End time formatted"] = pd.to_datetime(plot_data['End time'], unit='s')
    plot_data["Tour Duration formatted"] = pd.to_datetime(plot_data['Tour Duration'], unit='s')
    plot_data["Service Duration formatted"] = pd.to_datetime(plot_data['Service Duration'], unit='s')
    plot_data["Travel Duration formatted"] = pd.to_datetime(plot_data['Travel Duration'], unit='s')

    plot_data["Start time"] = pd.to_numeric(plot_data["Start time"])
    plot_data["End time"] = pd.to_numeric(plot_data["End time"])
    plot_data["Tour Duration"] = pd.to_numeric(plot_data["Tour Duration"])
    plot_data["Service Duration"] = pd.to_numeric(plot_data["Service Duration"])
    plot_data["Travel Duration"] = pd.to_numeric(plot_data["Travel Duration"])
    
    return plot_data , startG , endG

def parse_carriers_from_xml(root):
    """
    This function parses carriers from an XML root.
    
    Args:
        root (ElementTree): The XML root to parse carriers from.
        
    Returns:
        list: The list of carriers.
    """

    carriers = []
    totalVeh = 0
    ns = {"m": "http://www.matsim.org/files/dtd"}

    for carrierXML in root.findall("m:carrier", ns):
        carrierID = carrierXML.attrib.get("id")
        if "supply" in carrierID:
            continue

        newCarrier = Carrier(carrierID)

        for attributeXML in carrierXML.findall("m:attributes", ns):
            for attribute in attributeXML.findall("m:attribute", ns):
                if attribute.attrib.get("name") == "missedParcelDeliveriesAsString":
                    missedParcels = attribute.text
                    missedParcels = missedParcels.replace("[", "").replace("]", "").replace(" ", "")
                    missedServicePerCarrier = missedParcels.split(",")
                    newCarrier.setMissedDeliveries(missedServicePerCarrier)

        for serviceXML in carrierXML.findall(".//m:service", ns):
            serviceID = serviceXML.attrib.get("id")

            serviceCapacityDemand = int(serviceXML.attrib.get("capacityDemand"))
            serviceDuration = serviceXML.attrib.get("serviceDuration")
            serviceLink = serviceXML.attrib.get("to")

            known_keys = {"id", "capacityDemand", "serviceDuration", "to"}
            extra_attrs = {k: v for k, v in serviceXML.attrib.items() if k not in known_keys}

            attributes_element = serviceXML.find("m:attributes", ns)
            if attributes_element is not None:
                for attr in attributes_element.findall("m:attribute", ns):
                    key = attr.attrib.get("name")
                    value = attr.text
                    if key in ["b2b", "b2c"]:
                        try:
                            value = int(value)
                        except ValueError:
                            value = 0
                    extra_attrs[key] = value

            newService = Service(
                "service",
                serviceID,
                serviceCapacityDemand,
                serviceDuration,
                serviceLink,
                extra_attributes=extra_attrs
            )
            newCarrier.addService(newService)

        for plan in carrierXML.findall(".//m:plan", ns):
            if plan.attrib.get("selected") == "true":
                i = 0
                for tour in plan.findall(".//m:tour", ns):
                    i += 1
                    vehID = tour.attrib.get("vehicleId") + "_" + str(i)
                    vehicle = Vehicle(vehID, "cep")                    

                    b = 0
                    c = 0

                    activities = {}
                    legs = {}
                    routes = {}

                    for act in tour.findall(".//m:act", ns):
                        actType = act.attrib.get("type")
                        if actType == "start":
                            activities[actType] = act
                        elif actType == "end":
                            activities[actType] = act
                        else:
                            actID = act.attrib.get("serviceId")
                            activities[actID] = act

                    for leg in tour.findall(".//m:leg", ns):
                        legs["leg_" + str(b)] = leg
                        b += 1

                    for route in tour.findall(".//m:route", ns):
                        if route.text is None:
                            routes["route_" + str(c)] = None
                        else:
                            routes["route_" + str(c)] = route.text.split(" ")
                        c += 1

                    for key, value in legs.items():
                        routeKey = "route_" + str(key.split("_")[1])
                        route = routes.get(routeKey)
                        value.attrib["route"] = route

                    vehiclePlan = Plan("plan_" + vehID, vehicle, activities, legs)
                    vehiclePlan.createInternalPlanSequence()
                    vehicle.addVehiclePlan(vehiclePlan)
                    newCarrier.addVehicle(vehicle)

                totalVeh += newCarrier.getNumberOfVehicles()

        carriers.append(newCarrier)

    return carriers



def calculate_costs(row):
    veh_time_cost = 22.87 / 3600  # Kosten pro Sekunde

    if "size_l" in row['vehicle_id']:
        veh_cap, veh_fix, veh_km_cost = 230, 189.15, 0.386
    elif "size_m" in row['vehicle_id']: 
        veh_cap, veh_fix, veh_km_cost = 165, 171.78, 0.372
    else:
        if "supply_light_van" in row['vehicle_id']:
            veh_cap, veh_fix, veh_km_cost = 230, 189.15, 0.386
        elif "light" in row['vehicle_id']:
            veh_cap, veh_fix, veh_km_cost = 1000, 550.63, 0.48643
        else:
            veh_cap, veh_fix, veh_km_cost = 2000, 618.55, 0.555126     
      
     
    vehicle_fix_cost = veh_fix
    vehicle_km_cost = row["tour_km"] * veh_km_cost
    vehicle_time_cost = row["Tour Duration"] * veh_time_cost

    overtime_seconds = max(0, row["Tour Duration"] - (7.5 * 3600))
    overtime_cost = overtime_seconds * veh_time_cost

    vehicle_cost = vehicle_fix_cost + vehicle_km_cost + overtime_cost

    return pd.Series([vehicle_fix_cost, vehicle_km_cost, vehicle_time_cost, overtime_cost, vehicle_cost])


def add_vehicle_demand_to_result(carriers, result):
    
    """
    This function adds vehicle demand to the result DataFrame.
    
    Args:
        carriers (list):The list of carriers.
        result (DataFrame): The DataFrame to add vehicle demand to.
        
    Returns:
        DataFrame: The DataFrame with added vehicle demand.
    """

    required_cols = [
        'deliveries', 'missed deliveries', 'b2b_ration', 'b2c_ration',
        'ration_check', 'vehicle_load_factor', 'vehicle_deliver_factor',
        'vehicle_fix_cost', 'vehicle_km_cost', 'vehicle_time_cost',
        'overtime_cost', 'vehicle_cost', 'service_num'
    ]
    for col in required_cols:
        if col not in result.columns:
            result[col] = np.nan
    expected_ids = []
    
        
    veh = 0
    fail = 0

    # 1. Create a new dictionary to hold vehicle-service mappings
    vehicle_services_dict = {}

    for c in carriers:
        missedDeliveries = c.getMissedDeliveries()
        missed_deliveries_set = set(missedDeliveries) 

        services = c.getServices()

        print(f"🔍 Carrier {c.getCarrierId()} has {len(c.getVehicles())} vehicles")
        
        
        for k, v in c.getVehicles().items():
            veh = veh + 1 
            result_df_id = "freight_" + c.getCarrierId()+"_veh_"+v.getVehicleId()     
            expected_ids.append(result_df_id)   
         
            veh_df_res = result[result.vehicle_id == result_df_id]    
            if (len(veh_df_res) != 1):
                print(result_df_id)
                fail = fail + 1    
                
            mask = result.vehicle_id == result_df_id
            if mask.sum() == 0:
                print(f"⚠️ ID not found in result: {result_df_id}")   
            
            totalVehDemand = 0            
            missedParcels = 0

            b2b = 0
            b2c = 0

            # 2. Extract services for this vehicle
            veh_services = []
            services_dict = {s.getServiceID(): s for s in services}

            for a in v.getPlans()[0].activities:
                if "service" in a:
                    for s in services:
                        if a == s.getServiceID():
                            service = services_dict.get(a)
                            veh_services.append(service)

                            service_id = s.getServiceID()
                            service_b2b = int(s.getAttribute("b2b", 0))
                            service_b2c = int(s.getAttribute("b2c", 0))

                            # Nachfrage aufsummieren
                            b2b += service_b2b
                            b2c += service_b2c
                            serviceDemand = service_b2b + service_b2c
                            totalVehDemand += serviceDemand

                            # Missed Check
                            missed = False
                            merged = s.getAttribute("mergedMetadata", None)
                            
                            if service_id in missed_deliveries_set:
                                missed = True
                            elif merged:
                                try:
                                    merged_dict = json.loads(merged)

                                    # Validierung: mixed muss zu merged passen
                                    if "MIXED" not in service_id.upper():
                                        print(f"⚠️ Warning: mergedMetadata found, but service ID does not indicate MIXED: {service_id}")

                                    for sid in merged_dict:
                                        if sid in missed_deliveries_set:
                                            missed = True
                                            break
                                except Exception as e:
                                    print(f"⚠️ Error while parsing mergedMetadata for {service_id}: {e}")

                            if missed:
                                missedParcels += serviceDemand  
                                    
            vehicle_services_dict[result_df_id] = veh_services                         
            veh_cap = 230 if "size_l" in result_df_id else 165

            b2b_ration = b2b / totalVehDemand
            b2c_ration = b2c / totalVehDemand
            
            result.loc[result.vehicle_id == result_df_id, 'b2b_ration'] = b2b_ration
            result.loc[result.vehicle_id == result_df_id, 'b2c_ration'] = b2c_ration
            result.loc[result.vehicle_id == result_df_id, 'ration_check'] = b2b_ration + b2c_ration

            result.loc[result.vehicle_id == result_df_id, 'deliveries'] = totalVehDemand
            if result[result.vehicle_id == result_df_id].empty:
                print(f"Assignment failed for: {result_df_id}")
            
            result.loc[result.vehicle_id == result_df_id, 'missed deliveries'] = round(missedParcels,0)
            
             # Anwenden der Funktion auf den DataFrame
            cols = ['vehicle_fix_cost', 'vehicle_km_cost', 'vehicle_time_cost', 'overtime_cost', 'vehicle_cost']
            result[cols] = result.apply(calculate_costs, axis=1)
             

            result.loc[result.vehicle_id == result_df_id, 'vehicle_load_factor'] = totalVehDemand / veh_cap
            result.loc[result.vehicle_id == result_df_id, 'vehicle_deliver_factor'] = (veh_cap - missedParcels) / veh_cap
    
    validate_vehicle_id_assignment(result, expected_ids)
    
    result['deliveries_per_stop'] = result['deliveries'] / result['service_num']
    result['Hannover'] = result['vehicle_id'].apply(lambda x: any(plz in x for plz in plzList))

    result['services'] = result['vehicle_id'].map(vehicle_services_dict)
    
    return result

def validate_vehicle_id_assignment(result, expected_ids):
    """
    Validates whether all expected vehicle IDs are present in the result DataFrame.
    Prints helpful diagnostics for debugging.
    
    Args:
        result (DataFrame): The result DataFrame after add_vehicle_demand_to_result.
        expected_ids (list): List of vehicle_id strings that should exist in result.
    """
    actual_ids = result['vehicle_id'].astype(str).tolist()

    missing_ids = [vid for vid in expected_ids if vid not in actual_ids]
    duplicate_ids = result['vehicle_id'][result['vehicle_id'].duplicated()].unique().tolist()
    empty_ids = result['vehicle_id'].isna().sum()

    print("\n📋 Vehicle ID Validation Report")
    print("──────────────────────────────")
    print(f"🔢 Expected IDs total: {len(expected_ids)}")
    print(f"✅ Found: {len(expected_ids) - len(missing_ids)}")
    print(f"❌ Missing: {len(missing_ids)}")
    print(f"🔁 Duplicates: {len(duplicate_ids)}")
    print(f"⚠️ Empty vehicle_id entries: {empty_ids}")
    
    if missing_ids:
        print("\n❌ Missing IDs (first 10 shown):")
        for mid in missing_ids[:10]:
            print(f"  - {mid}")
    
    if duplicate_ids:
        print("\n🔁 Duplicate vehicle_ids:")
        for did in duplicate_ids:
            print(f"  - {did}")



In [5]:
def format_runtime(duration):
    """Format the runtime duration in either seconds, milliseconds, or minutes."""
    if duration < 1:  # If less than 1 second
        return f"{duration * 1000:.2f} ms"
    elif duration < 60:  # If less than 1 minute
        return f"{duration:.2f} s"
    else:  # If 1 minute or more
        minutes = duration // 60
        seconds = duration % 60
        return f"{int(minutes)} min {seconds:.2f} s"
    
# This is a constant representing the number of seconds in a day.
DAYEND = 24*3600

# This is a list of postal codes (it seems) that the script will be working with.
plzList = ["30159", "30161", "30163", "30165", "30167", "30169", "30171", "30173", "30175", "30177", "30179", "30419", "30449"
           ,"30451" ,"30453" ,"30455" ,"30457" ,"30459" ,"30519" ,"30521" ,"30539" ,"30559" ,"30625" ,"30627" ,"30629",
           "30625" , "30627", "30629", "30655", "30657", "30659", "30669", "31303", "31303"]

# This is the path to the folder where input files are stored.
# folderPath = "Input/SimRes/500-it/basecase_13052025/"
folderPath = "Input/SimRes/BC_500_it_no_reduction/"

file = ""

# This is the name of the file where event data will be loaded.
eventFileName = "basecase_13052025.output_events.xml.gz"

# This is the name of the file where carrier data will be loaded.
carrierFileName = "basecase_13052025.output_carriers.xml.gz"

city = regionclusters[regionclusters.raumtyp < 7]

result_dataframes = {}
result_networks= {}

event_file = folderPath + file + eventFileName
carrier_file = folderPath + file + carrierFileName

# Begin script execution
print(f"[{datetime.datetime.now()}] [1/10] Initiating script execution...")
overall_start_time = time.time()

print(f"[{datetime.datetime.now()}] [2/10] Loading events from {event_file}...")
start_time = time.time()
vehicle_tour, service_events, network_volumes, nr_events = parse_events(event_file)
print(f"[{datetime.datetime.now()}] [INFO] Loaded events for {len(vehicle_tour)} vehicles. (Runtime: {format_runtime(time.time() - start_time)})")
print(f"[{datetime.datetime.now()}] [INFO] Number of parsed events: {nr_events}")
print(f"[{datetime.datetime.now()}] [INFO] Network size: {len(network_volumes)}")

print(f"[{datetime.datetime.now()}] [3/10] Processing network volumes...")
start_time = time.time()
clipped_network_volumes = gpd.clip(network_volumes, gdf_areas)
end_time = time.time()
reduced_size = len(clipped_network_volumes)
reduction_percentage = (1 - (reduced_size / len(network_volumes))) * 100

print(f"[{datetime.datetime.now()}] [INFO] Network volumes processed. (Runtime: {format_runtime(end_time - start_time)})")
print(f"[{datetime.datetime.now()}] [INFO] Reduced Network size to: {reduced_size} ({reduction_percentage:.2f}% reduction)")
      
print(f"[{datetime.datetime.now()}] [4/10] Calculating vehicle statistics...")
start_time = time.time()
veh_df = vehicle_stats(vehicle_tour, service_events)
veh_df_van = veh_df[veh_df.veh_class == "van"]
veh_df_truck = veh_df[~(veh_df.veh_class == "van")]
print(f"[{datetime.datetime.now()}] [INFO] Statistics calculated for {len(veh_df)} vehicles (Vans: {len(veh_df_van)}, Trucks: {len(veh_df_truck)}). (Runtime: {format_runtime(time.time() - start_time)})")

print(f"[{datetime.datetime.now()}] [5/10] Processing vehicle data...")
start_time = time.time()
veh_df = process_vehicle_data(veh_df)
print(f"[{datetime.datetime.now()}] [INFO] Vehicle data processed. (Runtime: {format_runtime(time.time() - start_time)})")

print(f"[{datetime.datetime.now()}] [6/10] Extracting vehicle information...")
start_time = time.time()
vehicles = get_vehicles(vehicle_tour)
print(f"[{datetime.datetime.now()}] [INFO] Extracted data for {len(vehicles)} vehicles. (Runtime: {format_runtime(time.time() - start_time)})")

print(f"[{datetime.datetime.now()}] [7/10] Creating plot data...")
start_time = time.time()
plot_data, startG, endG = create_plot_data(vehicles, event_file)
print(f"[{datetime.datetime.now()}] [INFO] Plot data created. (Runtime: {format_runtime(time.time() - start_time)})")

print(f"[{datetime.datetime.now()}] [8/10] Integrating plot data with vehicle data...")
start_time = time.time()
result = plot_data.merge(veh_df, left_on='vehicle_id', right_on='vehicle_id')
print(f"[{datetime.datetime.now()}] [INFO] Data integrated. (Runtime: {format_runtime(time.time() - start_time)})")

print(f"[{datetime.datetime.now()}] [9/10] Parsing carriers from XML at {carrier_file}...")
start_time = time.time()
with gzip.open(carrier_file, mode="rt") as f:
    tree = ET.parse(f)
    root = tree.getroot()
carriers = parse_carriers_from_xml(root)
print(f"[{datetime.datetime.now()}] [INFO] Carriers parsed. (Runtime: {format_runtime(time.time() - start_time)})")

print(f"[{datetime.datetime.now()}] [10/10] Augmenting result data with vehicle demand information...")
start_time = time.time()
result = add_vehicle_demand_to_result(carriers, result)
print(f"[{datetime.datetime.now()}] [INFO] Script execution completed successfully. (Total Runtime: {format_runtime(time.time() - overall_start_time)})")


[2025-07-04 16:42:32.327945] [1/10] Initiating script execution...
[2025-07-04 16:42:32.328234] [2/10] Loading events from Input/SimRes/BC_500_it_no_reduction/basecase_13052025.output_events.xml.gz...
[2025-07-04 16:42:41.604676] [INFO] Loaded events for 2015 vehicles. (Runtime: 9.28 s)
[2025-07-04 16:42:41.605105] [INFO] Number of parsed events: 1367146
[2025-07-04 16:42:41.605150] [INFO] Network size: 544514
[2025-07-04 16:42:41.605189] [3/10] Processing network volumes...
[2025-07-04 16:42:55.251606] [INFO] Network volumes processed. (Runtime: 13.65 s)
[2025-07-04 16:42:55.252005] [INFO] Reduced Network size to: 77916 (85.69% reduction)
[2025-07-04 16:42:55.252105] [4/10] Calculating vehicle statistics...
[2025-07-04 16:42:57.273816] [INFO] Statistics calculated for 2015 vehicles (Vans: 1738, Trucks: 277). (Runtime: 2.02 s)
[2025-07-04 16:42:57.274015] [5/10] Processing vehicle data...
[2025-07-04 16:42:57.276402] [INFO] Vehicle data processed. (Runtime: 2.36 ms)
[2025-07-04 16:42:5

C:\Users\bienzeisler\AppData\Local\Temp\ipykernel_4824\3511914712.py:587: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  plot_data["Start time"][df_start.iloc[event]["person"]] = df_start.iloc[event]['time']
C:\Users\bienzeisler\AppData\Local

[2025-07-04 16:43:25.825222] [INFO] Plot data created. (Runtime: 28.55 s)
[2025-07-04 16:43:25.825433] [8/10] Integrating plot data with vehicle data...
[2025-07-04 16:43:25.828513] [INFO] Data integrated. (Runtime: 3.05 ms)
[2025-07-04 16:43:25.828573] [9/10] Parsing carriers from XML at Input/SimRes/BC_500_it_no_reduction/basecase_13052025.output_carriers.xml.gz...
[2025-07-04 16:43:34.988495] [INFO] Carriers parsed. (Runtime: 9.16 s)
[2025-07-04 16:43:34.988718] [10/10] Augmenting result data with vehicle demand information...
🔍 Carrier dhl_30926_1 has 8 vehicles
🔍 Carrier dhl_30926_0 has 8 vehicles
🔍 Carrier amazon_31535_0 has 7 vehicles
🔍 Carrier dpd_30890 has 6 vehicles
🔍 Carrier dpd_30419 has 4 vehicles
🔍 Carrier dpd_30539 has 2 vehicles
🔍 Carrier dpd_30659 has 2 vehicles
🔍 Carrier dpd_30657 has 3 vehicles
🔍 Carrier dpd_30655 has 2 vehicles
🔍 Carrier amazon_31535_1 has 7 vehicles
🔍 Carrier hermes_31535_0 has 5 vehicles
🔍 Carrier gls_31535 has 7 vehicles
🔍 Carrier gls_30449 has 2

In [6]:
from tqdm import tqdm


vehicles_list = result['vehicle_id'].unique()
time_columns = [f"{h:02}:{m:02}" for h in range(24) for m in range(60)]  # Generate time columns for every minute
vehicle_status_df = pd.DataFrame(-3, index=vehicles_list, columns=time_columns)

for name, group_start in tqdm(startG, desc="Processing vehicle tours", unit="tour"):
    
#     if "supply" in name:
#         continue
    
    num_services = result[result.vehicle_id == name]["service_num"].iloc[0]
    num_deliveries = result[result.vehicle_id == name]["deliveries"].iloc[0]
    
    tour_duration = result[result.vehicle_id == name]["Tour Duration"].iloc[0]
    
    services = result[result.vehicle_id == name]["services"].iloc[0]    

    # Assuming that "Start time formatted" is already in a Timestamp format. If not, convert it.
    start_time_tour = pd.to_datetime(result[result.vehicle_id == name]["Start time formatted"].iloc[0])
    end_time_tour = pd.to_datetime(result[result.vehicle_id == name]["End time formatted"].iloc[0])
    
    loading_start = start_time_tour - pd.Timedelta(minutes=60)
    
    loading_start_minute = loading_start.round('min')
    start_time_tour_minute = start_time_tour.round('min')
    end_time_tour_minute = end_time_tour.round('min')
    
        # Check if loading_start is before start_time_tour
    if loading_start_minute >= start_time_tour_minute:
        raise Exception(f"For vehicle {name}, loading start time {loading_start_minute} is not before start time {start_time_tour_minute}.")

    # Check if start_time_tour is before end_time_tour
    if start_time_tour_minute >= end_time_tour_minute:
        raise Exception(f"For vehicle {name}, start time {start_time_tour_minute} is not before end time {end_time_tour_minute}.")

    # Check if loading_start is approximately 30 minutes before start_time_tour
    time_difference = (start_time_tour_minute - loading_start_minute).seconds / 60
    if not (29 <= time_difference <= 61):  # Allowing 1 minute buffer due to rounding
        raise Exception(f"For vehicle {name}, the difference between loading start {loading_start_minute} and start time {start_time_tour_minute} is not approximately 30 minutes. Actual difference: {time_difference} minutes.")

    
    # Convert the rounded timestamps to HH:MM string format   
    loading_start_str = loading_start_minute.strftime('%H:%M')    
    loading_end_str = (start_time_tour_minute - pd.Timedelta(minutes=1)).strftime('%H:%M')    
    
    start_time_tour_str = start_time_tour_minute.strftime('%H:%M')
    end_time_tour_str = end_time_tour_minute.strftime('%H:%M')

    # Check if the times exist as columns in vehicle_status_df
    if (loading_start_str in vehicle_status_df.columns) and (start_time_tour_str in vehicle_status_df.columns):
        vehicle_status_df.loc[name, loading_start_str:start_time_tour_str] = -2  # Loading        

    if (start_time_tour_str in vehicle_status_df.columns) and (end_time_tour_str in vehicle_status_df.columns):
        vehicle_status_df.loc[name, start_time_tour_str:end_time_tour_str] = -1  # Driving
    
    veh_size = result[result.vehicle_id == name]["veh_size"].iloc[0]

    # Set expected services based on vehicle size
    if veh_size == "l":
        expected_services = 230
    elif veh_size == "m":
        expected_services = 165
    elif  veh_size == "supply_van":
        expected_services = 230
    elif veh_size == "truck_light":
        expected_services = 1000        
    elif veh_size == "truck":
        expected_services = 2000
    else:
        raise Exception(f"Unexpected vehicle size: {veh_size}")

    if num_services != (len(group_start) ):
        raise Exception("Number of services does not match the expected value.")
        
    # Fetch and display corresponding group from endG
    group_end = endG.get_group(name)
    
    # Concatenate the dataframes
    combined_df = pd.concat([group_start, group_end])

    # Sort by the 'time' column
    sorted_df = combined_df.sort_values(by='time')
    
    startUtilization = (num_deliveries / expected_services) * 100    
    
        # Creating a new 'group_id' column where every two rows have the same ID
    sorted_df['group_id'] = [i//2 for i, _ in enumerate(sorted_df.index)]

    # Now, you can group by 'group_id' to get pairs of rows
    grouped = [group for _, group in sorted_df.groupby('group_id')]
    
    current_load = num_deliveries

    for index, g in enumerate(grouped):
        correspondingService = services[index]

        if len(g["link"].unique()) > 1:
            raise Exception("Number of links for group does not match the expected value.")
        if correspondingService.getServiceLink() != g["link"].unique()[0]:
            raise Exception("Both Links are not matching for found Services", correspondingService.getServiceLink(), g["link"].unique()[0])

        currentDemand = correspondingService.getDemand()

        start = pd.to_datetime(g.iloc[0]["time"], unit='s').round('min')
        end = pd.to_datetime(g.iloc[1]["time"], unit='s').round('min')

        # Convert start and end to HH:MM format
        start_str = start.strftime('%H:%M')
        end_str = end.strftime('%H:%M')

        # Calculate and round load percentage for the current state of the vehicle
        load_percentage = round((current_load / expected_services) * 100)

        # Update vehicle_status_df with the load percentage during the service activity
        if (start_str in vehicle_status_df.columns) and (end_str in vehicle_status_df.columns):
            vehicle_status_df.loc[name, start_str:end_str] = load_percentage 

        # Reduce the current load by the deliveries made in this service activity
        current_load -= currentDemand

Processing vehicle tours: 100%|██████████| 1738/1738 [01:01<00:00, 28.32tour/s]


In [7]:
vans = result[result.veh_class == "van"]
supplyTrucks = result[result['veh_class'].isin(['truck', 'truck_light'])]

network_volumes_filtered = network_volumes[network_volumes.total_count > 0]

In [8]:
# vehicles_list = result['vehicle_id'].unique()
# time_columns = [f"{h:02}:{m:02}" for h in range(24) for m in range(60)]  # Generate time columns for every minute
# vehicle_status_df = pd.DataFrame(-3, index=vehicles_list, columns=time_columns)

def calc_co2_emissions(distance_m, vehicle_type, segment_type, load_percentage):
    
#     print(distance_m , vehicle_type, segment_type, load_percentage)
    """
    Calculate the CO2 emissions based on the distance, vehicle type, segment type, and load percentage.
    
    Args:
    - distance_m (float): Distance in meters.
    - vehicle_type (str): Type of the vehicle. (e.g., "Van [< 2t]", "Truck [< 10t]")
    - segment_type (str): Road segment type. (e.g., "Urban", "Rural", "Highway")
    - load_percentage (float): Load percentage of the vehicle. (0-100, where 100 is full and 0 is empty)
    
    Returns:
    - float: CO2 emissions in grams.
    """
    
    emission_factors = {
        "van [< 2t]": {"urban": (197, 213), "rural": (117, 126), "highway": (171, 185)},
        "van [> 2t]": {"urban": (276, 302), "rural": (170, 186), "highway": (250, 275)},
        "truck [< 10t]": {"urban": (419, 472), "rural": (281, 316), "highway": (253, 286)},
        "truck [10-20 t]": {"urban": (858, 1009), "rural": (555, 653), "highway": (471, 554)},
        "truck [10-20 t] + trailer": {"urban": (1023, 1387), "rural": (658, 892), "highway": (559, 757)},
        "truck [> 20t]": {"urban": (1223, 1501), "rural": (791, 871), "highway": (649, 796)},
        "truck [> 20t] + trailer": {"urban": (1321, 1922), "rural": (838, 1220), "highway": (700, 1019)},
        "tractor-trailer light": {"urban": (1189, 1565), "rural": (768, 1011), "highway": (619, 814)},
        "tractor-trailer heavy": {"urban": (1447, 2258), "rural": (901, 1407), "highway": (628, 981)},
        "longer heavier vehicle": {"urban": (1954, 3048), "rural": (1217, 1899), "highway": (848, 1324)}
    }
    
        # Check if vehicle type is None or not in emission factors
    if vehicle_type not in emission_factors:
        print(f"Vehicle type '{vehicle_type}' not found in emission factors.")
    # If vehicle type is valid, check if segment type is valid for this vehicle type
    elif segment_type not in emission_factors[vehicle_type]:
        print(f"Segment type '{segment_type}' not found for vehicle type '{vehicle_type}' in emission factors.")
    # If both types are valid, proceed with the calculation
    else:
    
        # Extract the emission factors for empty and full loads
        empty, full = emission_factors[vehicle_type][segment_type]

        # Linearly interpolate based on the load percentage
        interpolated_emission = empty + (full - empty) * (load_percentage / 100)

        # Calculate the CO2 emissions
        emissions = (distance_m / 1000) * interpolated_emission
    
        return emissions


def convert_seconds_to_timestamp(seconds):
    """
    Convert seconds since the start of the day to a rounded Timestamp format.

    Parameters:
    - seconds: int, time in seconds since the start of the day (can be passed as string).

    Returns:
    - pd.Timestamp rounded to the nearest minute.
    """
    # Ensure input is numeric (int or float)
    seconds = float(seconds)

    # Convert the seconds to a timedelta
    time_timedelta = pd.to_timedelta(seconds, unit='s')
    
    # Set a reference starting point (start of the day)
    reference_time = pd.Timestamp('00:00:00')
    
    # Add the timedelta to the reference to get the time in Timestamp format
    time_formatted = reference_time + time_timedelta    
    
    # Round it to the nearest minute
    time_rounded = time_formatted.round('min')    

    return time_rounded


def get_vehicle_size_from_df(vehicle_entry):

    
    # Extract the vehicle size from the result dataframe
    vehSize_df = vehicle_entry['veh_size'].values[0]
    
    # Map the vehicle size from the result dataframe to the corresponding label in the dictionary
    vehSize_mapped = vehicle_size_mapping.get(vehSize_df, None)

    return vehSize_mapped


def find_next_positive_load(vehId, start_time_str, vehicle_status_df):
    
    # Convert the string timestamp to a pd.Timestamp object
    current_time = pd.Timestamp(start_time_str) + pd.Timedelta(minutes=1)

    # Loop until a value different from -1 is found or we reach the end of the dataframe columns
    while current_time.strftime('%H:%M') in vehicle_status_df.columns:
        # Check the value for the current timestamp
        load_value = vehicle_status_df.loc[vehId, current_time.strftime('%H:%M')]

        # If the load value is different from -1, handle accordingly
        if load_value != -1:
            if load_value > 0:
                return load_value
            else:
                return 0

        # Otherwise, increment the timestamp by one minute
        current_time += pd.Timedelta(minutes=1)

    # If no value different from -1 is found, raise an exception
    raise Exception(f"No value different from -1 found for vehicle {vehId} after time {start_time_str}")


# Create a mapping dictionary
vehicle_size_mapping = {
    'm': 'van [< 2t]',
    'l': 'van [> 2t]',
    'truck': 'truck [10-20 t] + trailer',
    'truck_light': 'truck [< 10t]',
    'supply_light_van': 'van [> 2t]',
}

# Initialize a dictionary to hold total emissions for each vehicle size
emissions_by_size = {
    'van [< 2t]': 0,
    'van [> 2t]': 0,
    'truck [10-20 t] + trailer': 0,
    'truck [< 10t]': 0,
    # Add other vehicle sizes if needed
}

# Total emissions across all vehicles
total_emissions = 0

# Define the time intervals
time_intervals = [f"{h:02}:{m:02}" for h in range(24) for m in [0, 15, 30, 45]]

link_ids = network_volumes_filtered['link_id'].unique()
# Initialize the nested dictionary
# emissions_dict = {link_id: {timestamp: 0 for timestamp in time_intervals} for link_id in link_ids}
# Initialize the nested dictionary
emissions_dict = {size: {link_id: {timestamp: 0 for timestamp in time_intervals} 
                         for link_id in link_ids} 
                  for size in emissions_by_size.keys()}

# 1. Initialize the dictionary
vehicle_emissions_dict = {}

for i, (vehId, tour) in enumerate(vehicle_tour.items()):
    if i % 100 == 0:
        print(f"[{i}] Processing vehicle ID: {vehId} | Number of total tours: {len(vehicle_tour)}")
    
    vehicle_entry = result[result['vehicle_id'] == vehId] 
    if vehicle_entry.empty:
        continue

    vehSize = get_vehicle_size_from_df(vehicle_entry)
    
    emissions = 0

    for linkInfo in tour:    

        linkId = linkInfo[0]
        linkTime = linkInfo[1]
        
        linkTime_formatted_str = convert_seconds_to_timestamp(linkTime)         
        linkTime_formatted_str_for_interval = linkTime_formatted_str.strftime('%H:%M')
        
        # Get the 15-min interval for this timestamp
        hour, minute = map(int, linkTime_formatted_str_for_interval.split(":"))
        interval = f"{hour:02}:{(minute // 15) * 15:02}"  # Floor the minutes to the nearest 15        
        
        # dict mit Netzwerk Link Längen
        linkLenth = link_length.get(linkId)
        linkType = link_type.get(linkId)
        
        # Find the next positive load value
        current_load = find_next_positive_load(vehId, linkTime_formatted_str, vehicle_status_df)
#             print(f"Current load for vehicle {vehId} at time {linkTime_formatted_str}: {current_load}")

        # print(f"Processing Link ID: {linkId} | Link Length: {linkLenth} | Vehicle Size: {vehSize} | Segment Type: {linkType} | Load Percentage: {current_load}%")
        # Test the function
        linkEmissions = calc_co2_emissions(linkLenth, vehSize, linkType, current_load) 
        # print(f"Link ID: {linkId} | Link Length: {linkLenth} m | Vehicle Size: {vehSize} | Segment Type: {linkType} | Load Percentage: {current_load}% | Emissions: {linkEmissions/1000:.2f} kg")
        emissions = emissions + linkEmissions
        
        # Update the emissions in the dictionary
        emissions_dict[vehSize][linkId][interval] += linkEmissions
        
    # Accumulate emissions for this vehicle size
    emissions_by_size[vehSize] += emissions
    total_emissions += emissions
    vehicle_emissions_dict[vehId] = emissions
    
#     print(f"Vehicle ID: {vehId} | Total Emissions: {emissions/1000:.2f} kg")

# 3. Map the dictionary to the result dataframe to create the new column
result['emissions'] = result['vehicle_id'].map(vehicle_emissions_dict)

emissions_df = pd.DataFrame.from_dict(emissions_dict, orient='index')

# # For each hour, sum the emissions for the four 15-minute intervals
# for hour in range(24):
#     emissions_df[f'sum_{hour:02}:00'] = emissions_df[[f'{hour:02}:00', f'{hour:02}:15', f'{hour:02}:30', f'{hour:02}:45']].sum(axis=1)

# # Calculate the total CO2 emissions for each link (sum of all intervals)
# # emissions_df['total_co2'] = emissions_df.sum(axis=1)
# emissions_df['total_co2'] = emissions_df[[f'sum_{hour:02}:00' for hour in range(24)]].sum(axis=1)


# # Merge with clipped_network_volumes_filtered
# merged_df = pd.merge(network_volumes_filtered, emissions_df, left_on='link_id', right_index=True)

# # Print results
# print(f"Total Emissions across all vehicles: {total_emissions/1000:.2f} kg")
# for size, emissions in emissions_by_size.items():
#     print(f"Total Emissions for {size}: {emissions/1000:.2f} kg")     
    
        
        

    


[0] Processing vehicle ID: freight_supply_fedex_hannover_north_veh_supply_fedex_hannover_north_supply_early_3 | Number of total tours: 2015
[100] Processing vehicle ID: freight_gls_30826_veh_cep_size_m_7_2 | Number of total tours: 2015
[200] Processing vehicle ID: freight_gls_30938_veh_cep_size_m_7_2 | Number of total tours: 2015
[300] Processing vehicle ID: freight_hermes_30916_veh_cep_size_m_7_3 | Number of total tours: 2015
[400] Processing vehicle ID: freight_dpd_30655_veh_cep_size_m_7_1 | Number of total tours: 2015
[500] Processing vehicle ID: freight_fedex_30179_veh_cep_size_m_7_2 | Number of total tours: 2015
[600] Processing vehicle ID: freight_dhl_30179_veh_cep_size_m_7_1 | Number of total tours: 2015
[700] Processing vehicle ID: freight_dhl_30880_1_veh_cep_size_l_7_7 | Number of total tours: 2015
[800] Processing vehicle ID: freight_dhl_30419_0_veh_cep_size_l_8_6 | Number of total tours: 2015
[900] Processing vehicle ID: freight_dhl_30890_1_veh_cep_size_m_8_3 | Number of tot

In [ ]:
emissions_df